# NorthStar Urban Mobility and Logistics
## Python Data Loading and Cleaning

My notebook will consist of  loading and cleaning each dataset before i continue the analysis tasks using SQL in R and MongoDB.

The approach in each section is to:
- load the csv
- inspect the dataset and its structure to find inconsistencies.
- check missing values and duplicates
- inspect category values
- apply cleaning steps
- save a cleaned csv that i can use for querying.

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display

base_url = "https://raw.githubusercontent.com/TaylorPendred/Datasets-and-Notebooks/main/"

orders = pd.read_csv(base_url + "orders.csv")
deliveries = pd.read_csv(base_url + "deliveries.csv")
customers = pd.read_csv(base_url + "customers.csv")
complaints = pd.read_csv(base_url + "complaints.csv")
incidents = pd.read_csv(base_url + "incidents.csv")
drivers = pd.read_csv(base_url + "drivers.csv")
vehicles = pd.read_csv(base_url + "vehicles.csv")
hubs = pd.read_csv(base_url + "hubs.csv")
app_events = pd.read_csv(base_url + "app_events.csv")
data_dictionary = pd.read_csv(base_url + "data_dictionary.csv")

## Case Study Task Focus

This cleaning notebook prepares the NorthStar datasets for later analysis and work on:
- service delays and failed operations
- customer dissatisfaction and complaint patterns
- variation in performance across zones and service types
- operational inefficiencies linked to hubs, incidents, drivers, and vehicles
- redesigning flexible operational records into MongoDB

## 1. Inspecting and Cleaning `orders.csv`

In [3]:
print("\nShape:", orders.shape)
display(orders.head(10))
print(list(orders.columns))
orders.info()


Shape: (1250, 11)


,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
0,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
1,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
2,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
3,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
4,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0
5,O00006,C0437,Retail,2024-08-05 04:55:00,1,CENTRAL,East,High,151.44,Web,1
6,O00007,C0001,Business,2024-05-05 21:32:00,2,CENTRAL,Airport,Low,76.12,App,0
7,O00008,C0157,Parcel,2024-04-03 17:54:00,4,Riverside,Riverside,Medium,35.06,Phone,0
8,O00009,C0141,Retail,2024-10-03 23:49:00,12,NORTH,East,Critical,78.93,App,0
9,O00010,C0171,Retail,2025-01-29 00:42:00,6,South,north,Low,34.55,Phone,0


['order_id', 'customer_id', 'service_type', 'order_created_at', 'promised_window_hours', 'pickup_zone', 'dropoff_zone', 'priority_level', 'order_value', 'booking_channel', 'special_handling_flag']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1250 entries, 0 to 1249
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id               1250 non-null   object 
 1   customer_id            1250 non-null   object 
 2   service_type           1250 non-null   object 
 3   order_created_at       1250 non-null   object 
 4   promised_window_hours  1250 non-null   int64  
 5   pickup_zone            1250 non-null   object 
 6   dropoff_zone           1250 non-null   object 
 7   priority_level         1250 non-null   object 
 8   order_value            1250 non-null   float64
 9   booking_channel        1225 non-null   object 
 10  special_handling_flag  1250 non-null   int64  
dtypes: float64(1), 

In [4]:
print("Missing values before cleaning:")
print(orders.isnull().sum())

print("\nFull duplicate rows:")
print(orders.duplicated().sum())

print("\nDuplicate order_id values:")
print(orders["order_id"].duplicated().sum())

Missing values before cleaning:
order_id                  0
customer_id               0
service_type              0
order_created_at          0
promised_window_hours     0
pickup_zone               0
dropoff_zone              0
priority_level            0
order_value               0
booking_channel          25
special_handling_flag     0
dtype: int64

Full duplicate rows:
0

Duplicate order_id values:
0


In [5]:
other_columns = ["service_type", "pickup_zone", "dropoff_zone", "priority_level", "booking_channel"]

for col in other_columns:
    print(f"\n {col}")
    print(sorted(orders[col].dropna().astype(str).unique()))


 service_type
['Business', 'Medical', 'Parcel', 'Passenger', 'Retail']

 pickup_zone
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']

 dropoff_zone
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']

 priority_level
['Critical', 'High', 'Low', 'Medium']

 booking_channel
['API', 'App', 'Phone', 'Web']


### Observation

The main issues in `orders.csv` are:
- inconsistent zone labels where duplicate names are used but with uppercase or lowercase such as `AIRPORT`, `Airport`, `north`, and `Ctr`
- missing values in `booking_channel`
- `order_created_at` needs to be converted to datetime

The category columns were inspected first before deciding what to clean.

In [6]:
orders.columns = orders.columns.str.strip()

text_cols = orders.select_dtypes(include="object").columns
for col in text_cols:
    orders[col] = orders[col].str.strip()

orders["pickup_zone"] = orders["pickup_zone"].str.title()
orders["dropoff_zone"] = orders["dropoff_zone"].str.title()

orders["pickup_zone"] = orders["pickup_zone"].replace("Ctr", "Central")
orders["dropoff_zone"] = orders["dropoff_zone"].replace("Ctr", "Central")

orders["order_created_at"] = pd.to_datetime(orders["order_created_at"], errors="coerce")

orders["booking_channel"] = orders["booking_channel"].fillna("Unknown")

orders = orders.drop_duplicates()

### Cleaning

- whitespace was removed from text fields
- `pickup_zone` and `dropoff_zone` were standardised to consistent case
- `Ctr` was standardised to `Central`
- missing `booking_channel` values were replaced with `Unknown` as its safer and more time efficient rather than spending hours predicting the correct value
- `order_created_at` was converted to datetime
- full duplicate rows were removed

In [7]:
print("Missing values after cleaning:")
print(orders.isnull().sum())

print("\nUpdated category values:")
for col in other_columns:
    print(f"\n {col}")
    print(sorted(orders[col].dropna().astype(str).unique()))

print("\nUpdated data types:")
print(orders.dtypes)

orders.to_csv("cleaned_orders.csv", index=False)
print("\nSaved cleaned_orders.csv")

Missing values after cleaning:
order_id                 0
customer_id              0
service_type             0
order_created_at         0
promised_window_hours    0
pickup_zone              0
dropoff_zone             0
priority_level           0
order_value              0
booking_channel          0
special_handling_flag    0
dtype: int64

Updated category values:

 service_type
['Business', 'Medical', 'Parcel', 'Passenger', 'Retail']

 pickup_zone
['Airport', 'Central', 'East', 'North', 'Riverside', 'South', 'West']

 dropoff_zone
['Airport', 'Central', 'East', 'North', 'Riverside', 'South', 'West']

 priority_level
['Critical', 'High', 'Low', 'Medium']

 booking_channel
['API', 'App', 'Phone', 'Unknown', 'Web']

Updated data types:
order_id                         object
customer_id                      object
service_type                     object
order_created_at         datetime64[ns]
promised_window_hours             int64
pickup_zone                      object
dropoff_zone    

---
## 2. Inspecting and Cleaning `deliveries.csv`

In [8]:
print("Shape:", deliveries.shape)
display(deliveries.head(10))
print(list(deliveries.columns))
deliveries.info()

Shape: (950, 13)


,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
0,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05
1,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41
2,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51
3,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62
4,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22
5,DL00006,O00029,D037,V098,H03,2024-09-11 12:40:00,2024-09-12 17:11:52.384869,Delayed,13.84,0,0,1.57,9.58
6,DL00007,O00097,D151,V037,H07,2024-01-09 13:41:00,2024-01-10 23:39:11.072850,Delayed,32.72,0,0,4.64,17.70
7,DL00008,O00207,D082,V066,H03,2024-08-22 21:34:00,2024-08-22 23:22:21.531162,OnTime,7.16,1,0,3.76,11.66
8,DL00009,O00297,D088,V029,H05,2024-04-12 21:33:00,2024-04-13 01:18:52.673288,OnTime,40.23,1,0,3.70,15.78
9,DL00010,O00836,D058,V057,H08,2025-09-22 19:09:00,2025-09-23 01:15:29.151459,Failed,9.85,1,0,3.20,9.31


['delivery_id', 'order_id', 'driver_id', 'vehicle_id', 'hub_id', 'dispatch_time', 'delivery_completed_at', 'delivery_status', 'route_distance_km', 'manual_route_override_count', 'proof_of_completion_missing', 'customer_rating_post_delivery', 'fuel_or_charge_cost']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   delivery_id                    950 non-null    object 
 1   order_id                       950 non-null    object 
 2   driver_id                      950 non-null    object 
 3   vehicle_id                     950 non-null    object 
 4   hub_id                         950 non-null    object 
 5   dispatch_time                  950 non-null    object 
 6   delivery_completed_at          931 non-null    object 
 7   delivery_status                950 non-null    object 
 8   route_distance_km        

In [9]:
print("Missing values before cleaning:")
print(deliveries.isnull().sum())

print("\nFull duplicate rows:")
print(deliveries.duplicated().sum())

print("\nDuplicate delivery_id values:")
print(deliveries["delivery_id"].duplicated().sum())

Missing values before cleaning:
delivery_id                       0
order_id                          0
driver_id                         0
vehicle_id                        0
hub_id                            0
dispatch_time                     0
delivery_completed_at            19
delivery_status                   0
route_distance_km                 0
manual_route_override_count       0
proof_of_completion_missing       0
customer_rating_post_delivery    14
fuel_or_charge_cost               0
dtype: int64

Full duplicate rows:
0

Duplicate delivery_id values:
0


In [10]:
categorical_cols = deliveries.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"\n {col} ")
    print(sorted(deliveries[col].dropna().astype(str).unique()))


 delivery_id 
['DL00001', 'DL00002', 'DL00003', 'DL00004', 'DL00005', 'DL00006', 'DL00007', 'DL00008', 'DL00009', 'DL00010', 'DL00011', 'DL00012', 'DL00013', 'DL00014', 'DL00015', 'DL00016', 'DL00017', 'DL00018', 'DL00019', 'DL00020', 'DL00021', 'DL00022', 'DL00023', 'DL00024', 'DL00025', 'DL00026', 'DL00027', 'DL00028', 'DL00029', 'DL00030', 'DL00031', 'DL00032', 'DL00033', 'DL00034', 'DL00035', 'DL00036', 'DL00037', 'DL00038', 'DL00039', 'DL00040', 'DL00041', 'DL00042', 'DL00043', 'DL00044', 'DL00045', 'DL00046', 'DL00047', 'DL00048', 'DL00049', 'DL00050', 'DL00051', 'DL00052', 'DL00053', 'DL00054', 'DL00055', 'DL00056', 'DL00057', 'DL00058', 'DL00059', 'DL00060', 'DL00061', 'DL00062', 'DL00063', 'DL00064', 'DL00065', 'DL00066', 'DL00067', 'DL00068', 'DL00069', 'DL00070', 'DL00071', 'DL00072', 'DL00073', 'DL00074', 'DL00075', 'DL00076', 'DL00077', 'DL00078', 'DL00079', 'DL00080', 'DL00081', 'DL00082', 'DL00083', 'DL00084', 'DL00085', 'DL00086', 'DL00087', 'DL00088', 'DL00089', 'DL00

### Observation

`deliveries.csv` contains missing values in `delivery_completed_at` and `customer_rating_post_delivery`.
These were not filled immediately because they may be meaningful.
For example, a delivery that failed may not have a completion time or a post-delivery rating.

In [11]:
deliveries.columns = deliveries.columns.str.strip()

text_cols = deliveries.select_dtypes(include="object").columns
for col in text_cols:
    deliveries[col] = deliveries[col].str.strip()

deliveries["dispatch_time"] = pd.to_datetime(deliveries["dispatch_time"], errors="coerce")
deliveries["delivery_completed_at"] = pd.to_datetime(deliveries["delivery_completed_at"], errors="coerce")

invalid_completion = deliveries["delivery_completed_at"].notna() & (deliveries["delivery_completed_at"] < deliveries["dispatch_time"])
print("Rows where delivery_completed_at is earlier than dispatch_time:", invalid_completion.sum())

deliveries.loc[invalid_completion, "delivery_completed_at"] = pd.NaT

deliveries = deliveries.drop_duplicates()

Rows where delivery_completed_at is earlier than dispatch_time: 64


In [12]:
print("Missing completion time by delivery status:")
print(deliveries.loc[deliveries["delivery_completed_at"].isnull(), "delivery_status"].value_counts(dropna=False))

print("\nMissing customer rating by delivery status:")
print(deliveries.loc[deliveries["customer_rating_post_delivery"].isnull(), "delivery_status"].value_counts(dropna=False))

Missing completion time by delivery status:
delivery_status
OnTime     76
Delayed     4
Failed      3
Name: count, dtype: int64

Missing customer rating by delivery status:
delivery_status
OnTime     8
Delayed    5
Failed     1
Name: count, dtype: int64


### Cleaning

- whitespace was removed from text fields
- `dispatch_time` and `delivery_completed_at` were converted to datetime
- rows where `delivery_completed_at` occurred before `dispatch_time` were treated as invalid and set to missing
- genuinely missing `delivery_completed_at` and `customer_rating_post_delivery` values were kept as missing because they may reflect failed or incomplete deliveries
- full duplicate rows were removed

In [13]:
print("Missing values after cleaning:")
print(deliveries.isnull().sum())

print("\nUpdated data types:")
print(deliveries.dtypes)

deliveries.to_csv("cleaned_deliveries.csv", index=False)
print("\nSaved cleaned_deliveries.csv")

Missing values after cleaning:
delivery_id                       0
order_id                          0
driver_id                         0
vehicle_id                        0
hub_id                            0
dispatch_time                     0
delivery_completed_at            83
delivery_status                   0
route_distance_km                 0
manual_route_override_count       0
proof_of_completion_missing       0
customer_rating_post_delivery    14
fuel_or_charge_cost               0
dtype: int64

Updated data types:
delivery_id                              object
order_id                                 object
driver_id                                object
vehicle_id                               object
hub_id                                   object
dispatch_time                    datetime64[ns]
delivery_completed_at            datetime64[ns]
delivery_status                          object
route_distance_km                       float64
manual_route_override_count        

---
## 3. Inspecting and Cleaning `customers.csv`

In [14]:
print("Shape:", customers.shape)
display(customers.head(10))
print(list(customers.columns))
customers.info()

Shape: (650, 9)


,customer_id,age,home_zone,customer_type,signup_date,loyalty_score,app_engagement_score,preferred_channel,account_status
0,C0001,26,North,SME,2024-11-27 04:25:00,44.9,69.2,App,Active
1,C0002,61,AIRPORT,Consumer,2025-10-28 01:04:00,55.4,66.6,App,Active
2,C0003,66,East,Consumer,2025-07-02 03:23:00,75.9,33.8,NaN,Active
3,C0004,75,CENTRAL,Consumer,2025-08-19 01:58:00,32.5,33.0,App,Active
4,C0005,26,Riverside,Consumer,2025-06-03 06:02:00,55.9,100.0,Web,Active
5,C0006,41,WEST,Consumer,2024-03-29 13:26:00,39.9,43.3,Web,Active
6,C0007,54,WEST,Consumer,2024-03-12 07:05:00,36.1,39.0,App,Active
7,C0008,70,north,SME,2025-07-18 13:03:00,84.6,65.2,Web,Active
8,C0009,34,South,Consumer,2025-08-02 03:14:00,62.6,40.8,App,Active
9,C0010,23,West,Consumer,2025-10-07 20:37:00,87.2,48.6,Phone,Active


['customer_id', 'age', 'home_zone', 'customer_type', 'signup_date', 'loyalty_score', 'app_engagement_score', 'preferred_channel', 'account_status']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customer_id           650 non-null    object 
 1   age                   650 non-null    int64  
 2   home_zone             650 non-null    object 
 3   customer_type         650 non-null    object 
 4   signup_date           650 non-null    object 
 5   loyalty_score         630 non-null    float64
 6   app_engagement_score  650 non-null    float64
 7   preferred_channel     637 non-null    object 
 8   account_status        650 non-null    object 
dtypes: float64(2), int64(1), object(6)
memory usage: 45.8+ KB


In [15]:
print("Missing values before cleaning:")
print(customers.isnull().sum())

print("\nFull duplicate rows:")
print(customers.duplicated().sum())

print("\nDuplicate customer_id values:")
print(customers["customer_id"].duplicated().sum())

Missing values before cleaning:
customer_id              0
age                      0
home_zone                0
customer_type            0
signup_date              0
loyalty_score           20
app_engagement_score     0
preferred_channel       13
account_status           0
dtype: int64

Full duplicate rows:
0

Duplicate customer_id values:
0


In [16]:
other_columns = ["home_zone", "customer_type", "preferred_channel", "account_status"]

for col in other_columns:
    print(f"\n {col} ")
    print(sorted(customers[col].dropna().astype(str).unique()))


 home_zone 
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']

 customer_type 
['Consumer', 'Enterprise', 'SME']

 preferred_channel 
['App', 'Partner API', 'Phone', 'Web']

 account_status 
['Active', 'Dormant', 'Suspended']


### Observation

`customers.csv` contains missing values in `loyalty_score` and `preferred_channel`.
The zone column also shows the same case inconsistencies seen in the operational datasets.

In [17]:
customers.columns = customers.columns.str.strip()

text_cols = customers.select_dtypes(include="object").columns
for col in text_cols:
    customers[col] = customers[col].str.strip()

customers["home_zone"] = customers["home_zone"].str.title()
customers["home_zone"] = customers["home_zone"].replace("Ctr", "Central")

customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")

customers["preferred_channel"] = customers["preferred_channel"].fillna("Unknown")

customers = customers.drop_duplicates()

### Cleaning

- whitespace was removed from text fields
- `home_zone` was standardised to consistent case
- `Ctr` was standardised to `Central`
- `signup_date` was converted to datetime
- missing `preferred_channel` values were replaced with `Unknown`
- missing `loyalty_score` values were kept as missing because they cannot be recovered from this dataset alone
- full duplicate rows were removed

In [18]:
print("Missing values after cleaning:")
print(customers.isnull().sum())

print("\nUpdated category values:")
for col in other_columns:
    print(f"\n {col} ")
    print(sorted(customers[col].dropna().astype(str).unique()))

customers.to_csv("cleaned_customers.csv", index=False)
print("\nSaved cleaned_customers.csv")

Missing values after cleaning:
customer_id              0
age                      0
home_zone                0
customer_type            0
signup_date              0
loyalty_score           20
app_engagement_score     0
preferred_channel        0
account_status           0
dtype: int64

Updated category values:

 home_zone 
['Airport', 'Central', 'East', 'North', 'Riverside', 'South', 'West']

 customer_type 
['Consumer', 'Enterprise', 'SME']

 preferred_channel 
['App', 'Partner API', 'Phone', 'Unknown', 'Web']

 account_status 
['Active', 'Dormant', 'Suspended']

Saved cleaned_customers.csv


---
## 4. Inspecting and Cleaning `complaints.csv`

In [19]:
print("Shape:", complaints.shape)
display(complaints.head(10))
print(list(complaints.columns))
complaints.info()

Shape: (320, 10)


,complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
0,CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11,23.99
1,CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4,21.64
2,CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16,26.41
3,CP0004,C0631,O00406,Delay,App,Medium,2025-01-14 13:07:00,AwaitingCustomer,7,23.44
4,CP0005,C0535,O00154,Delay,Email,Medium,2024-08-31 05:56:00,Resolved,1,16.18
5,CP0006,C0096,O00147,Delay,App,Medium,2024-07-22 07:43:00,Resolved,9,18.51
6,CP0007,C0597,O00981,MissedPickup,App,Medium,2025-01-15 21:57:00,Escalated,4,14.14
7,CP0008,C0309,O00902,AppIssue,Email,High,2024-09-26 19:41:00,Resolved,18,NaN
8,CP0009,C0340,O00011,Delay,App,Low,2025-08-16 04:19:00,Resolved,4,26.35
9,CP0010,C0486,O00417,DriverBehaviour,Phone,Medium,2025-01-13 15:33:00,Open,1,6.32


['complaint_id', 'customer_id', 'order_id', 'complaint_type', 'channel', 'severity', 'created_at', 'status', 'resolution_days', 'compensation_amount']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   complaint_id         320 non-null    object 
 1   customer_id          320 non-null    object 
 2   order_id             320 non-null    object 
 3   complaint_type       320 non-null    object 
 4   channel              320 non-null    object 
 5   severity             320 non-null    object 
 6   created_at           320 non-null    object 
 7   status               320 non-null    object 
 8   resolution_days      320 non-null    int64  
 9   compensation_amount  304 non-null    float64
dtypes: float64(1), int64(1), object(8)
memory usage: 25.1+ KB


In [20]:
print("Missing values before cleaning:")
print(complaints.isnull().sum())

print("\nFull duplicate rows:")
print(complaints.duplicated().sum())

print("\nDuplicate complaint_id values:")
print(complaints["complaint_id"].duplicated().sum())

Missing values before cleaning:
complaint_id            0
customer_id             0
order_id                0
complaint_type          0
channel                 0
severity                0
created_at              0
status                  0
resolution_days         0
compensation_amount    16
dtype: int64

Full duplicate rows:
0

Duplicate complaint_id values:
0


In [21]:
other_columns = ["complaint_type", "channel", "severity", "status"]

for col in other_columns:
    print(f"\n {col} ")
    print(sorted(complaints[col].dropna().astype(str).unique()))


 complaint_type 
['AppIssue', 'Billing', 'Damage', 'Delay', 'DriverBehaviour', 'MissedPickup', 'SupportExperience']

 channel 
['App', 'Chatbot', 'Email', 'Phone']

 severity 
['High', 'Low', 'Medium']

 status 
['AwaitingCustomer', 'Escalated', 'Open', 'Resolved']


### Observation

`complaints.csv` contains missing values in `compensation_amount`.
This is not automatically treated as bad data because not every complaint results in compensation.

In [22]:
complaints.columns = complaints.columns.str.strip()

text_cols = complaints.select_dtypes(include="object").columns
for col in text_cols:
    complaints[col] = complaints[col].str.strip()

complaints["created_at"] = pd.to_datetime(complaints["created_at"], errors="coerce")

complaints = complaints.drop_duplicates()

### Cleaning

- whitespace was removed from text fields
- `created_at` was converted to datetime
- missing `compensation_amount` values were kept as missing because no reliable replacement is available and the missingness may be meaningful
- full duplicate rows were removed

In [23]:
print("Missing values after cleaning:")
print(complaints.isnull().sum())

complaints.to_csv("cleaned_complaints.csv", index=False)
print("\nSaved cleaned_complaints.csv")

Missing values after cleaning:
complaint_id            0
customer_id             0
order_id                0
complaint_type          0
channel                 0
severity                0
created_at              0
status                  0
resolution_days         0
compensation_amount    16
dtype: int64

Saved cleaned_complaints.csv


---
## 5. Inspecting and Cleaning `incidents.csv`

In [24]:
print("Shape:", incidents.shape)
display(incidents.head(10))
print(list(incidents.columns))
incidents.info()

Shape: (280, 7)


,incident_id,delivery_id,incident_type,reported_at,severity,resolution_status,resolved_hours
0,I0001,DL00221,BatteryAlert,2024-03-11 23:46:00,Medium,Escalated,12.3
1,I0002,DL00578,BatteryAlert,2024-02-21 10:56:00,Low,Open,9.6
2,I0003,DL00175,TemperatureIssue,2025-04-17 23:22:00,Medium,Open,22.0
3,I0004,DL00417,ProofMissing,2025-02-09 00:16:00,Medium,Closed,9.8
4,I0005,DL00897,RouteDeviation,2025-01-04 02:49:00,Low,Open,13.0
5,I0006,DL00634,CustomerNoShow,2025-08-08 21:26:00,High,PendingVendor,19.9
6,I0007,DL00495,AppSyncError,2024-05-23 02:29:00,Low,PendingVendor,4.8
7,I0008,DL00602,RouteDeviation,2024-08-17 01:27:00,Medium,Closed,12.1
8,I0009,DL00636,CustomerNoShow,2024-08-24 22:28:00,Medium,Closed,15.8
9,I0010,DL00052,CustomerNoShow,2025-05-06 14:17:00,Medium,Closed,9.4


['incident_id', 'delivery_id', 'incident_type', 'reported_at', 'severity', 'resolution_status', 'resolved_hours']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   incident_id        280 non-null    object 
 1   delivery_id        280 non-null    object 
 2   incident_type      280 non-null    object 
 3   reported_at        280 non-null    object 
 4   severity           280 non-null    object 
 5   resolution_status  280 non-null    object 
 6   resolved_hours     263 non-null    float64
dtypes: float64(1), object(6)
memory usage: 15.4+ KB


In [25]:
print("Missing values before cleaning:")
print(incidents.isnull().sum())

print("\nFull duplicate rows:")
print(incidents.duplicated().sum())

print("\nDuplicate incident_id values:")
print(incidents["incident_id"].duplicated().sum())

Missing values before cleaning:
incident_id           0
delivery_id           0
incident_type         0
reported_at           0
severity              0
resolution_status     0
resolved_hours       17
dtype: int64

Full duplicate rows:
0

Duplicate incident_id values:
0


In [26]:
other_columns = ["incident_type", "severity", "resolution_status"]

for col in other_columns:
    print(f"\n {col} ")
    print(sorted(incidents[col].dropna().astype(str).unique()))


 incident_type 
['AppSyncError', 'BatteryAlert', 'CustomerNoShow', 'ProofMissing', 'RouteDeviation', 'SafetyNearMiss', 'TemperatureIssue', 'VehicleFault']

 severity 
['Critical', 'High', 'Low', 'Medium']

 resolution_status 
['Closed', 'Escalated', 'Open', 'PendingVendor']


### Observation

`incidents.csv` contains missing values in `resolved_hours`.
These may be expected when the incident is still open or escalated.

In [27]:
incidents.columns = incidents.columns.str.strip()

text_cols = incidents.select_dtypes(include="object").columns
for col in text_cols:
    incidents[col] = incidents[col].str.strip()

incidents["reported_at"] = pd.to_datetime(incidents["reported_at"], errors="coerce")

incidents = incidents.drop_duplicates()

### Cleaning

- whitespace was removed from text fields
- `reported_at` was converted to datetime
- missing `resolved_hours` values were kept as missing because they may reflect unresolved incidents
- full duplicate rows were removed

In [28]:
print("Missing values after cleaning:")
print(incidents.isnull().sum())

incidents.to_csv("cleaned_incidents.csv", index=False)
print("\nSaved cleaned_incidents.csv")

Missing values after cleaning:
incident_id           0
delivery_id           0
incident_type         0
reported_at           0
severity              0
resolution_status     0
resolved_hours       17
dtype: int64

Saved cleaned_incidents.csv


---
## 6. Inspecting and Cleaning `drivers.csv`

In [29]:
print("Shape:", drivers.shape)
display(drivers.head(10))
print(list(drivers.columns))
drivers.info()

Shape: (170, 8)


,driver_id,base_zone,employment_type,years_experience,training_score,driver_rating,shift_preference,active_flag
0,D001,AIRPORT,FullTime,8,67.8,3.54,Morning,1
1,D002,Central,FullTime,4,42.4,3.94,Evening,1
2,D003,AIRPORT,FullTime,11,96.5,5.00,Evening,1
3,D004,Airport,PartTime,13,88.9,4.75,Morning,1
4,D005,north,FullTime,3,69.7,4.14,Morning,1
5,D006,CENTRAL,FullTime,8,78.8,4.38,Flexible,1
6,D007,North,FullTime,4,92.6,3.94,Evening,1
7,D008,SOUTH,FullTime,9,84.1,3.88,Morning,1
8,D009,Airport,FullTime,15,63.4,3.80,Night,1
9,D010,West,FullTime,8,70.0,3.95,Evening,1


['driver_id', 'base_zone', 'employment_type', 'years_experience', 'training_score', 'driver_rating', 'shift_preference', 'active_flag']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   driver_id         170 non-null    object 
 1   base_zone         170 non-null    object 
 2   employment_type   170 non-null    object 
 3   years_experience  170 non-null    int64  
 4   training_score    163 non-null    float64
 5   driver_rating     170 non-null    float64
 6   shift_preference  170 non-null    object 
 7   active_flag       170 non-null    int64  
dtypes: float64(2), int64(2), object(4)
memory usage: 10.8+ KB


In [30]:
print("Missing values before cleaning:")
print(drivers.isnull().sum())

print("\nFull duplicate rows:")
print(drivers.duplicated().sum())

print("\nDuplicate driver_id values:")
print(drivers["driver_id"].duplicated().sum())

Missing values before cleaning:
driver_id           0
base_zone           0
employment_type     0
years_experience    0
training_score      7
driver_rating       0
shift_preference    0
active_flag         0
dtype: int64

Full duplicate rows:
0

Duplicate driver_id values:
0


In [31]:
other_columns = ["base_zone", "employment_type", "shift_preference"]

for col in other_columns:
    print(f"\n--- {col} ---")
    print(sorted(drivers[col].dropna().astype(str).unique()))


--- base_zone ---
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']

--- employment_type ---
['Contract', 'FullTime', 'PartTime']

--- shift_preference ---
['Evening', 'Flexible', 'Morning', 'Night']


### Observation

`drivers.csv` contains missing values in `training_score` and the same zone label inconsistencies found in the other files.

In [32]:
drivers.columns = drivers.columns.str.strip()

text_cols = drivers.select_dtypes(include="object").columns
for col in text_cols:
    drivers[col] = drivers[col].str.strip()

drivers["base_zone"] = drivers["base_zone"].str.title()
drivers["base_zone"] = drivers["base_zone"].replace("Ctr", "Central")

drivers = drivers.drop_duplicates()

### Cleaning

- whitespace was removed from text fields
- `base_zone` was standardised to consistent case
- `Ctr` was standardised to `Central`
- missing `training_score` values were kept as missing because they cannot be recovered reliably from this dataset alone
- full duplicate rows were removed

In [33]:
print("Missing values after cleaning:")
print(drivers.isnull().sum())

drivers.to_csv("cleaned_drivers.csv", index=False)
print("\nSaved cleaned_drivers.csv")

Missing values after cleaning:
driver_id           0
base_zone           0
employment_type     0
years_experience    0
training_score      7
driver_rating       0
shift_preference    0
active_flag         0
dtype: int64

Saved cleaned_drivers.csv


---
## 7. Inspecting and Cleaning `vehicles.csv`

In [34]:
print("Shape:", vehicles.shape)
display(vehicles.head(10))
print(list(vehicles.columns))
vehicles.info()

Shape: (120, 8)


,vehicle_id,vehicle_type,assigned_zone,commission_date,battery_health_pct,odometer_km,maintenance_status,telematics_version
0,V001,EV,North,2024-12-28 23:48:00,71.8,56928,Active,v2.2
1,V002,EV,AIRPORT,2024-04-21 16:14:00,67.9,159368,InRepair,v2.2
2,V003,CargoVan,north,2025-11-24 23:59:00,91.7,219359,Active,v2.1
3,V004,Hybrid,RiverSide,2024-06-07 13:21:00,NaN,36310,Active,v2.2
4,V005,CargoVan,West,2025-11-15 11:08:00,58.6,146638,Active,v2.2
5,V006,EV,Central,2025-11-22 06:39:00,78.6,141381,Active,v2.1
6,V007,Diesel,AIRPORT,2025-09-17 08:52:00,68.6,78468,Active,v2.2
7,V008,EV,RiverSide,2025-06-14 03:47:00,90.5,49711,Active,v2.0
8,V009,CargoVan,South,2025-05-01 08:50:00,68.8,156687,Active,v2.1
9,V010,CargoVan,NORTH,2025-10-07 22:40:00,50.7,129032,InRepair,v2.1


['vehicle_id', 'vehicle_type', 'assigned_zone', 'commission_date', 'battery_health_pct', 'odometer_km', 'maintenance_status', 'telematics_version']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   vehicle_id          120 non-null    object 
 1   vehicle_type        120 non-null    object 
 2   assigned_zone       120 non-null    object 
 3   commission_date     120 non-null    object 
 4   battery_health_pct  116 non-null    float64
 5   odometer_km         120 non-null    int64  
 6   maintenance_status  120 non-null    object 
 7   telematics_version  120 non-null    object 
dtypes: float64(1), int64(1), object(6)
memory usage: 7.6+ KB


In [35]:
print("Missing values before cleaning:")
print(vehicles.isnull().sum())

print("\nFull duplicate rows:")
print(vehicles.duplicated().sum())

print("\nDuplicate vehicle_id values:")
print(vehicles["vehicle_id"].duplicated().sum())

Missing values before cleaning:
vehicle_id            0
vehicle_type          0
assigned_zone         0
commission_date       0
battery_health_pct    4
odometer_km           0
maintenance_status    0
telematics_version    0
dtype: int64

Full duplicate rows:
0

Duplicate vehicle_id values:
0


In [36]:
other_columns = ["vehicle_type", "assigned_zone", "maintenance_status", "telematics_version"]

for col in other_columns:
    print(f"\n {col} ")
    print(sorted(vehicles[col].dropna().astype(str).unique()))


 vehicle_type 
['CargoVan', 'Diesel', 'EV', 'Hybrid']

 assigned_zone 
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']

 maintenance_status 
['Active', 'InRepair', 'Scheduled']

 telematics_version 
['v1.9', 'v2.0', 'v2.1', 'v2.2']


### Observation

`vehicles.csv` contains missing values in `battery_health_pct` and the same zone label inconsistencies found in the other operational files.

In [37]:
vehicles.columns = vehicles.columns.str.strip()

text_cols = vehicles.select_dtypes(include="object").columns
for col in text_cols:
    vehicles[col] = vehicles[col].str.strip()

vehicles["assigned_zone"] = vehicles["assigned_zone"].str.title()
vehicles["assigned_zone"] = vehicles["assigned_zone"].replace("Ctr", "Central")

vehicles["commission_date"] = pd.to_datetime(vehicles["commission_date"], errors="coerce")

vehicles = vehicles.drop_duplicates()

### Cleaning

- whitespace was removed from text fields
- `assigned_zone` was standardised to consistent case
- `Ctr` was standardised to `Central`
- `commission_date` was converted to datetime
- missing `battery_health_pct` values were kept as missing because they cannot be reliably inferred
- full duplicate rows were removed

In [38]:
print("Missing values after cleaning:")
print(vehicles.isnull().sum())

vehicles.to_csv("cleaned_vehicles.csv", index=False)
print("\nSaved cleaned_vehicles.csv")

Missing values after cleaning:
vehicle_id            0
vehicle_type          0
assigned_zone         0
commission_date       0
battery_health_pct    4
odometer_km           0
maintenance_status    0
telematics_version    0
dtype: int64

Saved cleaned_vehicles.csv


---
## 8. Inspecting and Cleaning `hubs.csv`

In [39]:
print("Shape:", hubs.shape)
display(hubs.head(10))
print(list(hubs.columns))
hubs.info()

Shape: (8, 5)


,hub_id,hub_name,zone,hub_type,capacity_score
0,H01,North Exchange,North,Dispatch,82
1,H02,South Link,South,Dispatch,78
2,H03,East Dock,East,Warehouse,74
3,H04,West Gate,West,Dispatch,69
4,H05,Central Core,Central,Control,88
5,H06,Airport Hub,Airport,Dispatch,71
6,H07,Riverside Hub,Riverside,Warehouse,66
7,H08,Midtown Relay,Central,Charging,63


['hub_id', 'hub_name', 'zone', 'hub_type', 'capacity_score']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   hub_id          8 non-null      object
 1   hub_name        8 non-null      object
 2   zone            8 non-null      object
 3   hub_type        8 non-null      object
 4   capacity_score  8 non-null      int64 
dtypes: int64(1), object(4)
memory usage: 452.0+ bytes


In [40]:
print("Missing values before cleaning:")
print(hubs.isnull().sum())

print("\nFull duplicate rows:")
print(hubs.duplicated().sum())

print("\nDuplicate hub_id values:")
print(hubs["hub_id"].duplicated().sum())

Missing values before cleaning:
hub_id            0
hub_name          0
zone              0
hub_type          0
capacity_score    0
dtype: int64

Full duplicate rows:
0

Duplicate hub_id values:
0


In [41]:
other_columns = ["hub_name", "zone", "hub_type"]

for col in other_columns:
    print(f"\n {col} ")
    print(sorted(hubs[col].dropna().astype(str).unique()))


 hub_name 
['Airport Hub', 'Central Core', 'East Dock', 'Midtown Relay', 'North Exchange', 'Riverside Hub', 'South Link', 'West Gate']

 zone 
['Airport', 'Central', 'East', 'North', 'Riverside', 'South', 'West']

 hub_type 
['Charging', 'Control', 'Dispatch', 'Warehouse']


In [42]:
hubs.columns = hubs.columns.str.strip()

text_cols = hubs.select_dtypes(include="object").columns
for col in text_cols:
    hubs[col] = hubs[col].str.strip()

hubs["zone"] = hubs["zone"].str.title()

hubs = hubs.drop_duplicates()

### Cleaning

---



`hubs.csv` is already relatively clean.
Only basic trimming and a final standardisation of the `zone` field were needed.

In [43]:
print("Missing values after cleaning:")
print(hubs.isnull().sum())

hubs.to_csv("cleaned_hubs.csv", index=False)
print("\nSaved cleaned_hubs.csv")

Missing values after cleaning:
hub_id            0
hub_name          0
zone              0
hub_type          0
capacity_score    0
dtype: int64

Saved cleaned_hubs.csv


---
## 9. Inspecting and Cleaning `app_events.csv`

In [44]:
print("Shape:", app_events.shape)
display(app_events.head(10))
print(list(app_events.columns))
app_events.info()

Shape: (640, 10)


,event_id,customer_id,order_id,event_timestamp,event_type,session_id,device_type,zone_context,api_latency_ms,success_flag
0,AE00001,C0488,NaN,2024-08-09 03:25:00,eta_refresh,S19847,Android,north,301,1
1,AE00002,C0595,O00950,2024-02-13 22:29:00,search_route,S32766,Android,SOUTH,60,1
2,AE00003,C0494,O00170,2025-08-11 09:29:00,chat_opened,S99516,iOS,Airport,1118,1
3,AE00004,C0407,O00756,2025-08-23 17:38:00,eta_refresh,S41236,iOS,CENTRAL,442,1
4,AE00005,C0506,NaN,2024-05-29 10:33:00,search_route,S12030,iOS,north,60,1
5,AE00006,C0498,O01196,2025-02-08 09:50:00,track_order,S66655,Web,West,499,1
6,AE00007,C0033,O00028,2025-04-20 19:06:00,chat_opened,S79495,Android,South,169,1
7,AE00008,C0091,O00076,2024-08-18 02:08:00,search_route,S59028,Android,Ctr,431,1
8,AE00009,C0421,O01222,2025-07-20 11:46:00,eta_refresh,S76526,Web,North,609,1
9,AE00010,C0553,O00854,2024-07-10 13:11:00,chat_opened,S24618,Android,WEST,167,1


['event_id', 'customer_id', 'order_id', 'event_timestamp', 'event_type', 'session_id', 'device_type', 'zone_context', 'api_latency_ms', 'success_flag']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 640 entries, 0 to 639
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   event_id         640 non-null    object
 1   customer_id      640 non-null    object
 2   order_id         496 non-null    object
 3   event_timestamp  640 non-null    object
 4   event_type       640 non-null    object
 5   session_id       640 non-null    object
 6   device_type      640 non-null    object
 7   zone_context     640 non-null    object
 8   api_latency_ms   640 non-null    int64 
 9   success_flag     640 non-null    int64 
dtypes: int64(2), object(8)
memory usage: 50.1+ KB


In [45]:
print("Missing values before cleaning:")
print(app_events.isnull().sum())

print("\nFull duplicate rows:")
print(app_events.duplicated().sum())

print("\nDuplicate event_id values:")
print(app_events["event_id"].duplicated().sum())

Missing values before cleaning:
event_id             0
customer_id          0
order_id           144
event_timestamp      0
event_type           0
session_id           0
device_type          0
zone_context         0
api_latency_ms       0
success_flag         0
dtype: int64

Full duplicate rows:
0

Duplicate event_id values:
0


In [46]:
other_columns = ["event_type", "device_type", "zone_context"]

for col in other_columns:
    print(f"\n {col} ")
    print(sorted(app_events[col].dropna().astype(str).unique()))


 event_type 
['cancel_attempt', 'chat_escalated', 'chat_opened', 'delivery_instruction_update', 'eta_refresh', 'payment_retry', 'search_route', 'track_order']

 device_type 
['Android', 'Web', 'iOS']

 zone_context 
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']


### Observation

`app_events.csv` contains missing values in `order_id`.
These are likely expected because not every app event must be tied to a specific order.
The zone context field also has the same label inconsistency seen elsewhere.

In [47]:
app_events.columns = app_events.columns.str.strip()

text_cols = app_events.select_dtypes(include="object").columns
for col in text_cols:
    app_events[col] = app_events[col].str.strip()

app_events["zone_context"] = app_events["zone_context"].str.title()
app_events["zone_context"] = app_events["zone_context"].replace("Ctr", "Central")

app_events["event_timestamp"] = pd.to_datetime(app_events["event_timestamp"], errors="coerce")

app_events = app_events.drop_duplicates()

### Cleaning

- whitespace was removed from text fields
- `zone_context` was standardised to consistent case
- `Ctr` was standardised to `Central`
- `event_timestamp` was converted to datetime
- missing `order_id` values were kept as missing because some app events may not be linked to an order
- full duplicate rows were removed

In [48]:
print("Missing values after cleaning:")
print(app_events.isnull().sum())

app_events.to_csv("cleaned_app_events.csv", index=False)
print("\nSaved cleaned_app_events.csv")

Missing values after cleaning:
event_id             0
customer_id          0
order_id           144
event_timestamp      0
event_type           0
session_id           0
device_type          0
zone_context         0
api_latency_ms       0
success_flag         0
dtype: int64

Saved cleaned_app_events.csv


---
## 10. Inspecting and Cleaning `data_dictionary.csv`

In [49]:
print("Shape:", data_dictionary.shape)
display(data_dictionary.head(10))
print(list(data_dictionary.columns))
data_dictionary.info()

Shape: (9, 3)


,file_name,record_count,description
0,hubs.csv,8,Operational hubs and control points
1,customers.csv,650,Customer master data with engagement and loyal...
2,drivers.csv,170,Driver workforce data with training and rating...
3,vehicles.csv,120,Fleet asset data including battery and mainten...
4,orders.csv,1250,Service orders across mobility and logistics o...
5,deliveries.csv,950,Operational dispatch and completion outcomes
6,incidents.csv,280,Delivery and asset incident records
7,complaints.csv,320,Customer complaints and compensation handling
8,app_events.csv,640,Digital platform events suitable for MongoDB r...


['file_name', 'record_count', 'description']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   file_name     9 non-null      object
 1   record_count  9 non-null      int64 
 2   description   9 non-null      object
dtypes: int64(1), object(2)
memory usage: 348.0+ bytes


In [50]:
print("Missing values before cleaning:")
print(data_dictionary.isnull().sum())

print("\nFull duplicate rows:")
print(data_dictionary.duplicated().sum())

Missing values before cleaning:
file_name       0
record_count    0
description     0
dtype: int64

Full duplicate rows:
0


In [51]:
data_dictionary.columns = data_dictionary.columns.str.strip()

text_cols = data_dictionary.select_dtypes(include="object").columns
for col in text_cols:
    data_dictionary[col] = data_dictionary[col].str.strip()

data_dictionary = data_dictionary.drop_duplicates()

### Cleaning

`data_dictionary.csv` is already clean from inspection. Only basic trimming and duplicate checking were needed.

In [52]:
print("Missing values after cleaning:")
print(data_dictionary.isnull().sum())

data_dictionary.to_csv("cleaned_data_dictionary.csv", index=False)
print("\nSaved cleaned_data_dictionary.csv")

Missing values after cleaning:
file_name       0
record_count    0
description     0
dtype: int64

Saved cleaned_data_dictionary.csv


# Saving Cleaned files as JSON files For mongoDB

In [53]:
def make_json(csv_file, json_file):
    df = pd.read_csv(csv_file)
    json_data = df.to_json(orient="records", indent=4)

    with open(json_file, "w") as f:
        f.write(json_data)

make_json("cleaned_orders.csv", "cleaned_orders.json")
make_json("cleaned_deliveries.csv", "cleaned_deliveries.json")
make_json("cleaned_customers.csv", "cleaned_customers.json")
make_json("cleaned_complaints.csv", "cleaned_complaints.json")
make_json("cleaned_incidents.csv", "cleaned_incidents.json")
make_json("cleaned_drivers.csv", "cleaned_drivers.json")
make_json("cleaned_vehicles.csv", "cleaned_vehicles.json")
make_json("cleaned_hubs.csv", "cleaned_hubs.json")
make_json("cleaned_app_events.csv", "cleaned_app_events.json")
make_json("cleaned_data_dictionary.csv", "cleaned_data_dictionary.json")

## End

At the end of this notebook, cleaned versions of all datasets are saved as new files with the new name `cleaned_`.
These cleaned files can be used in my SQL in R notebook and my MongoDB notebook.